### 2.2. 파이썬 내장 데이터베이스 연동 (`sqlite3`)

In [1]:
# 파이썬 내장 데이터베이스 연동 (`sqlite3`)

import sqlite3

# 1. 데이터베이스 연결 (파일이 없으면 현재 폴더에 새로 생성됨)
db_path = "my_app.db"
conn = sqlite3.connect(db_path)

try:
    # 2. 커서(Cursor) 객체 생성
    # 커서는 데이터베이스에 SQL 명령을 전달하고 결과를 가져오는 역할을 합니다.
    cursor = conn.cursor()

    # 3. 테이블 생성 (초기화)
    # IF NOT EXISTS: 테이블이 이미 존재할 경우 오류가 발생하는 것을 방지합니다.
    # AUTOINCREMENT: 데이터를 추가할 때마다 id 값이 1씩 자동으로 증가하도록 설정합니다.
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL
        )
    """)

    # 4. [Create] 데이터 삽입 (파라미터 바인딩 활용)
    # 주의: 문자열 포맷팅(f-string 등)을 직접 사용하면 SQL 인젝션 해킹에 취약해집니다.
    # 따라서 반드시 '?' 기호를 사용해 데이터를 안전하게 전달해야 합니다.
    insert_query = "INSERT INTO users (name) VALUES (?)"
    cursor.execute(insert_query, ("Alice",))

    # 5. 트랜잭션 확정
    # 지금까지의 변경 사항(테이블 생성, 데이터 삽입)을 데이터베이스에 영구적으로 반영합니다.
    conn.commit()
    print("데이터가 성공적으로 저장되었습니다.")

    # 6. [Read] 데이터 조회
    select_query = "Select * FROM users WHERE name = ?"
    cursor.execute(select_query, ("Alice",))

    # fetchall(): 조회된 모든 데이터를 리스트 안의 튜플 형태로 반환합니다.
    # fetchone(): 데이터 1건만 가져올 때 사용합니다.
    results = cursor.fetchall()
    print(f"조회 결과: {results}")  # 출력 예: [(1, 'Alice')]

except sqlite3.Error as e:
    # 7. 예외 처리 및 롤백
    # SQL 실행 중 오류가 발생하면, commit 되지 않은 모든 변경 사항을 취소(원상 복구)합니다.
    conn.rollback()
    print(f"데이터베이스 오류가 발생하여 작업을 취소했습니다: {e}")

finally:
    # 8. 자원 반환
    # 정상 종료되든 오류가 발생하든, 메모리 누수 방지를 위해 데이터베이스 연결은 반드시 닫아줍니다.
    conn.close()
    print("데이터베이스 연결을 안전하게 종료했습니다.")

데이터가 성공적으로 저장되었습니다.
조회 결과: [(1, 'Alice'), (2, 'Alice')]
데이터베이스 연결을 안전하게 종료했습니다.


### 2.3. 객체 관계 매핑 (ORM) 및 `SQLAlchemy` 2.0

In [2]:
#### [ 커넥션 풀링(Connection Pooling) 방어적 아키텍처 ]
"""
`create_engine()`은 내부적으로 `QueuePool`을 자동 생성하여 DB 커넥션을 재사용합니다.
실무에서는 트래픽 폭증 시 DB 서버 다운을 막기 위해 애플리케이션 단에서 연결 수를 제어해야 합니다.
"""

import os
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError

# 1. 데이터베이스 접속 정보 설정 (보안 강화)
# 실무에서는 비밀번호를 코드에 하드코딩하지 않고 환경 변수나 .env 파일을 활용합니다.
DB_USER = os.getenv("DB_USER", "user")
DB_PASS = os.getenv("DB_PASS", "pass")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_NAME = os.getenv("DB_NAME", "db")

# 접속 URL 포맷팅 (postgresql://사용자:비밀번호@호스트/데이터베이스)
DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}/{DB_NAME}"

# 2. 엔진(Engine) 생성 및 커넥션 풀(Connection Pool) 설정
engine = create_engine(
    DATABASE_URL,
    pool_size=10,       # [기본] 평상시 유지할 기본 커넥션 개수
    max_overflow=20,    # [추가] 트래픽 폭증 시 추가로 허용할 최대 커넥션 개수 (총 30개까지 가능)
    pool_timeout=30,    # [심화] 모든 커넥션이 사용 중일 때, 새 커넥션을 기다리는 최대 시간(초)
    pool_recycle=3600   # [심화] 1시간(3600초) 이상 된 낡은 커넥션을 닫고 새로 연결 (DB 자동 끊김 방지)
)

# 3. 연결 테스트 (학습용)
# 설정한 엔진이 데이터베이스와 정상적으로 통신하는지 확인합니다.
try:
    # engine.connect()를 with문과 함께 사용하면 작업 종료 시 커넥션을 풀에 자동으로 반납합니다.
    with engine.connect() as connection:
        # 간단한 쿼리를 실행하여 PostgreSQL의 버전을 가져옵니다.
        result = connection.execute(text("SELECT version();"))
        print(f"데이터베이스 연결 성공!\nPostgreSQL 버전: {result.fetchone()[0]}")

except SQLAlchemyError as e:
    print(f"데이터베이스 연결 실패: {e}")

데이터베이스 연결 실패: (psycopg2.OperationalError) connection to server at "localhost" (::1), port 5432 failed: Connection refused (0x0000274D/10061)
	Is the server running on that host and accepting TCP/IP connections?
connection to server at "localhost" (127.0.0.1), port 5432 failed: Connection refused (0x0000274D/10061)
	Is the server running on that host and accepting TCP/IP connections?

(Background on this error at: https://sqlalche.me/e/20/e3q8)


In [4]:
#### [ SQLAlchemy 최신 동기식 실전 활용 및 N+1 문제 시연 ]
"""
SQLAlchemy 2.0은 타입 힌팅을 적극 활용합니다.
트랜잭션 관리 시 `with Session.begin() as session:`을 활용하면 안전한 트랜잭션 스코프를 구성할 수 있습니다.
"""
from sqlalchemy import create_engine, select, ForeignKey
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship, sessionmaker, selectinload

# ==========================================
# 1. ORM 모델 정의 (SQLAlchemy 2.0 최신 문법)
# ==========================================
class Base(DeclarativeBase):
    pass

class User(Base):
    # 주의: 테이블 이름은 반드시 앞뒤 밑줄 두 개(__)를 붙인 __tablename__으로 지정해야 합니다.
    __tablename__ = "users"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str]
    
    # 1:N 관계 설정 (User는 여러 Order를 가질 수 있음)
    # back_populates를 통해 양방향 참조를 설정합니다.
    orders: Mapped[list["Order"]] = relationship(back_populates="user")

class Order(Base):
    __tablename__ = "orders"

    id: Mapped[int] = mapped_column(primary_key=True)
    item: Mapped[str]
    
    # 외래 키(Foreign Key) 설정: users 테이블의 id를 참조
    user_id: Mapped[int] = mapped_column(ForeignKey("users.id"))
    
    # N:1 관계 설정 (Order는 하나의 User에 속함)
    user: Mapped["User"] = relationship(back_populates="orders")


# ==========================================
# 2. 데이터베이스 연결 및 초기화
# ==========================================
# echo=True 설정 시 실행되는 실제 SQL 쿼리를 콘솔에서 확인할 수 있어 학습에 매우 유용합니다.
# sqlite:///orm_database.db (슬래시 3개 ///)는 상대 경로(Relative Path)를 의미
engine = create_engine("sqlite:///orm_database.db", echo=False)

# 등록된 모든 클래스(User, Order)를 바탕으로 데이터베이스에 테이블을 생성합니다.
Base.metadata.create_all(engine)

# Session 팩토리 생성
Session = sessionmaker(bind=engine)


# ==========================================
# 3. [Create] 안전한 트랜잭션 스코프 활용
# ==========================================
try:
    # Session.begin(): 내부 블록이 에러 없이 끝나면 자동 commit(), 예외 발생 시 자동 rollback()
    with Session.begin() as session:
        new_user = User(name="이순신")
        new_order1 = Order(item="거북선", user=new_user)
        new_order2 = Order(item="학익진 전술서", user=new_user) # 데이터 추가
        
        # add_all()을 통해 객체들을 세션에 추가합니다. 
        # (new_order들이 new_user를 참조하고 있으므로, SQLAlchemy가 알아서 순서에 맞게 INSERT 합니다.)
        session.add_all([new_user, new_order1, new_order2])
        print("데이터베이스에 성공적으로 저장되었습니다.")

except Exception as e:
    print(f"트랜잭션 실패 및 롤백 완료: {e}")


# ==========================================
# 4. [Read] 🚨 N+1 쿼리 문제 최적화 (Eager Loading)
# ==========================================
with Session() as session:
    print("\n[사용자 주문 내역 조회]")
    
    # 💡 selectinload를 통한 Eager Loading
    # 연관된 Order 데이터를 IN 절을 사용해 한 번의 추가 쿼리로 모두 끌어옵니다. (총 2번의 쿼리 발생)
    stmt = select(User).options(selectinload(User.orders))
    
    # scalars(): 튜플 형태가 아닌 ORM 객체 자체를 반환받기 위해 사용합니다.
    users = session.scalars(stmt).all()

    for u in users:
        # 이미 메모리에 로드되어 있으므로, u.orders에 접근할 때 추가 쿼리가 발생하지 않습니다.
        print(f"{u.name}의 주문 횟수: {len(u.orders)}건")
        for order in u.orders:
            print(f" ㄴ 주문 항목: {order.item}")
# Session 팩토리 생성
Session = sessionmaker(bind=engine)

# ==========================================
# 3. [Create] 안전한 트랜잭션 스코프 활용
# ==========================================
try:
    # Session.begin(): 내부 블록이 에러 없이 끝나면 자동 commit(), 예외 발생 시 자동 rollback()
    with Session.begin() as session:
        new_user = User(name="이순신")
        new_order1 = Order(item="거북선", user=new_user)
        new_order2 = Order(item="학익진 전술서", user=new_user) # 데이터 추가

        # add_all()을 통해 객체들을 세션에 추가합니다. 
        # (new_order들이 new_user를 참조하고 있으므로, SQLAlchemy가 알아서 순서에 맞게 INSERT 합니다.)
        session.add_all([new_user, new_order1, new_order2])
        print("데이터베이스에 성공적으로 저장되었습니다.")

except Exception as e:
    print(f"트랜잭션 실패 및 롤백 완료: {e}")

# ==========================================
# 4. [Read] 🚨 N+1 쿼리 문제 최적화 (Eager Loading)
# ==========================================
with Session() as session:
    print("\n[사용자 주문 내역 조회]")

    # 💡 selectinload를 통한 Eager Loading
    # 연관된 Order 데이터를 IN 절을 사용해 한 번의 추가 쿼리로 모두 끌어옵니다. (총 2번의 쿼리 발생)
    stmt = select(User).options(selectinload(User.orders))

    # scalars(): 튜플 형태가 아닌 ORM 객체 자체를 반환받기 위해 사용합니다.
    users = session.scalars(stmt).all()

    for u in users:
        # 이미 메모리에 로드되어 있으므로, u.orders에 접근할 때 추가 쿼리가 발생하지 않습니다.
        print(f"{u.name}의 주문 횟수: {len(u.orders)}건")
        for order in u.orders:
            print(f" ㄴ 주문 항목: {order.item}")

데이터베이스에 성공적으로 저장되었습니다.

[사용자 주문 내역 조회]
이순신의 주문 횟수: 2건
 ㄴ 주문 항목: 거북선
 ㄴ 주문 항목: 학익진 전술서
데이터베이스에 성공적으로 저장되었습니다.

[사용자 주문 내역 조회]
이순신의 주문 횟수: 2건
 ㄴ 주문 항목: 거북선
 ㄴ 주문 항목: 학익진 전술서
이순신의 주문 횟수: 2건
 ㄴ 주문 항목: 거북선
 ㄴ 주문 항목: 학익진 전술서


In [10]:
#### [ 최신 트렌드: 비동기 데이터베이스 연동 (Async I/O) ]

"""
FastAPI 같은 최신 파이썬 웹 프레임워크는 비동기 처리가 기본입니다.
DB 응답을 기다리는 동안 스레드가 블로킹되는 동기식 처리와 달리,
비동기 처리는 대기 시간 동안 다른 웹 요청을 처리할 수 있어 I/O 바운드 작업의 동시성을 극대화합니다.
"""

import asyncio
from sqlalchemy import select
from sqlalchemy.ext.asyncio import create_async_engine, async_sessionmaker, AsyncSession
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column

# ==========================================
# 1. ORM 모델 정의
# ==========================================
class Base(DeclarativeBase):
    pass

class User(Base):
    __tablename__ = "users"
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str]


# ==========================================
# 2. 비동기 엔진 및 세션 팩토리 생성
# ==========================================
# 드라이버 주의: sqlite3가 아닌 비동기 전용 드라이버 'aiosqlite'를 사용합니다.
# (PostgreSQL의 경우 asyncpg, MySQL의 경우 aiomysql 사용)
async_engine = create_async_engine("sqlite+aiosqlite:///async_db.db", echo=False)

# expire_on_commit=False: 비동기 환경에서 트랜잭션이 끝난 후 
# 객체에 다시 접근할 때 추가 쿼리가 발생하여 에러가 나는 것을 방지하는 필수 설정입니다.
AsyncSessionLocal = async_sessionmaker(
    bind=async_engine, 
    class_=AsyncSession, 
    expire_on_commit=False
)


# ==========================================
# 3. 비동기 함수 정의 (초기화 및 조회)
# ==========================================
async def init_db():
    """데이터베이스 테이블을 생성하는 초기화 함수"""
    async with async_engine.begin() as conn:
        # Base.metadata.create_all은 동기(Sync) 방식의 함수입니다.
        # 비동기 연결(conn) 내에서 동기 함수를 실행하려면 반드시 run_sync()로 감싸야 합니다.
        await conn.run_sync(Base.metadata.create_all)

async def get_users() -> list[User]:
    """유저 목록을 조회하는 비동기 함수"""
    # async with: 비동기 컨텍스트 매니저 (작업 완료 후 세션 자동 반환)
    async with AsyncSessionLocal() as session:
        # await: 데이터베이스에서 결과를 가져올 때까지 '기다림'을 선언. 
        # 이때 파이썬은 멈춰있지 않고 다른 사용자의 요청(I/O)을 처리할 수 있습니다.
        result = await session.scalars(select(User))
        return result.all()


# ==========================================
# 4. 실행 진입점 (Event Loop)
# ==========================================
async def main():
    await init_db()
    users = await get_users()
    print(f"조회된 유저 수: {len(users)}명")

if __name__ == "__main__":
    # 파이썬 비동기 코드는 반드시 asyncio.run()을 통해 이벤트 루프 위에서 실행해야 합니다.
    # asyncio.run(main()) -> .py 환경에서 사용할 것.
    await main()

조회된 유저 수: 0명
